# 第5章 benchmark 与可信计时

**操作手册** | 热身、重复、GPU event、避免测量陷阱

本手册对应文档：`docs/part1-profiling/chapter5/index.md`  
本手册对应代码：`code/part1-profiling/chapter5/`

---

## 本章导读

> 前面几章我们把环境、GPU 体系结构和第一个 vector add 程序串了起来，第 4 章还用一次 baseline benchmark 建立了「算子离 Roofline 上限有多远」的直觉。从这一篇（Part 1 profiling 篇）开始，问题变成：**怎么知道量出来的数字准不准、可不可信、会不会骗人？**
>
> 本章先把镜头对准「量准」这件事本身。Latency、Throughput、Bandwidth、FLOPS 这些指标，memory-bound / compute-bound 这套判断语言，以及 Roofline 心智模型，第 2–3 章和第 4 章已经建立，本章不再重复——而是把它们底下那层更基础的东西讲清楚：**一次 benchmark 怎样才算可信**。

读完本章后你应该能：
1. 解释为什么不能只跑一次就下结论
2. 说明 warmup、repeat、synchronize 这些「无聊」的细节如何决定数字是否可信
3. 用 GPU event 而不是 CPU wall clock 量出 kernel 的真实耗时
4. 用一份检查清单避免常见的伪优化

第 4 章你已经写过一个小 benchmark（`benchmark_vector_add.py`），里面用了 warmup、repeat 和 GPU event——但当时只是照着做。本章把每个步骤背后的「为什么」讲清楚。

## Goal

学会设计可信 benchmark，避免伪优化陷阱。具体目标：

1. 理解 warmup、repeat、synchronize 的必要性
2. 使用 GPU event 而非 wall clock 量准 kernel 时间
3. 输出 min / median / mean / p95 / std 统计量
4. 执行三段骨架：PyTorch op、Triton kernel、HIP event 计时
5. 掌握一份可信 benchmark 的检查清单，挡掉大部分伪优化

## Prerequisite

- 已完成第4章 baseline benchmark
- ROCm 环境已激活（`source code/part1-profiling/activate-rocm.sh`）
- PyTorch + Triton 可用
- 理解第2-4章的 Roofline 心智模型

## Platform

本手册基于以下环境验证：

- **GPU**: AMD Radeon RX 9070 XT (gfx1201)
- **ROCm**: 7.13
- **OS**: Ubuntu 24.04 (native, kernel 6.17.0-35-generic)
- **PyTorch**: 2.11.0+rocm7.13.0
- **Triton**: 3.2.0+rocm7.13.0

其他 RDNA3/RDNA4 架构（gfx1100, gfx1151, gfx1201）均可运行，数字会有差异。

## 5.1 为什么不能凭感觉优化

性能优化最怕的不是「没优化成功」，而是**你以为自己优化成功了，其实只是测错了**。

GPU 程序通常是异步执行的，第一次运行可能包含初始化或编译开销，同一个输入规模下也可能因为后台负载、缓存状态、调度方式而波动。只看一次运行时间，很容易把偶然现象当成规律。

很多刚开始做性能优化的同学，会直接问：「怎么把 GPU 跑满？」这个问题还不够具体——GPU 没跑满可能是数据没送到，可能是任务太碎，可能是测量本身不可靠，也可能它**确实**已经撞上了硬件天花板。

所以 Part 1 的第一步不是「马上优化」，而是先学会问更好的问题：**我到底在测什么？这个数字可信吗？它说明瓶颈在哪一层？**

下面这张表列出几种常见的「直觉判断」与它们更值得追问的问题：

| 直觉判断 | 可能的问题 | 更好的追问 |
| ---- | ---- | ---- |
| GPU 利用率低，所以 kernel 写得差 | 可能是 CPU 调度慢、输入太小、数据搬运多 | GPU 到底在等什么？任务有没有持续送进去？ |
| 改完代码以后快了一点 | 可能只是 warmup、缓存、后台负载或随机波动 | 重复测了吗？统计口径一样吗？ |
| 单个算子快了，端到端就会快 | 这个算子可能只占总时间的一小部分 | 它在整条链路里占多少比例？ |
| 平均时间下降了 | 可能尾延迟变差，或者波动变大 | median、min、p95、p99 有没有一起看？ |
| GPU 时间很短，说明程序很快 | 可能只量到了 CPU 提交任务的时间 | 计时前后有没有 synchronize？ |
| fp16 比 fp32 快两倍 | 也许只是计算路径变了，访存没变 | 是真省了带宽，还是只在算力侧变快？ |

**没有稳定测量，就没有可靠优化。**

*（图示：从「感觉慢」到「可验证优化」的最小闭环）*

优化不是从改代码开始，而是从**定义问题**和**设计测量**开始：

> **感觉它很慢** → **定义要优化的指标** → **设计可信 benchmark** → **收集 profiling 证据** → **提出优化假设** → **只改一个变量** → 再回到设计 benchmark 验证 → **结论可复查**

否则你很容易进入一种状态：代码改了很多，数字也变了，但没人知道到底是哪一步起作用。

## 5.2 热身与缓存效应

benchmark 里最常见的一个现象：**第一次总是特别慢**。

GPU 程序的「首轮开销」通常来自三件事：

1. **编译 / JIT**：第一次启动某个 kernel 时，驱动或框架可能还要做最终编译、链接、内核选择（Triton autotune、rocBLAS 的 kernel 选择都在这一步发生）
2. **缓存准备**：第一次访存时 L2、页表、TLB 都是冷的，数据要真正从显存搬进来
3. **时钟爬升**：GPU 的实际工作频率可能从空闲状态逐步爬升到标称频率，前几次还没爬满

如果直接拿第一次的时间当成绩，你量到的多半是这些一次性开销，而不是 kernel 真正的稳态性能。

*（图示：首轮的一次性开销会在 warmup 后消失，正式计时只看稳态段）*

> **第 1 次**（编译 + 冷缓存 + 爬频）→ **第 2-3 次**（缓存渐热）→ **warmup 后**（稳态）→ 正式计时从这开始 → **repeat N 次**

更隐蔽的是缓存带来的「伪提升」：第二次运行同一个输入往往比第一次快很多，原因只是数据已经被 L2 / GDDR6 预热，而不是你的代码变好了。

**应对方法：**

- **warmup**：正式计时前先空跑若干次（经验上至少 5~20 次），让 JIT、cache、时钟进入稳定状态
- **按工作集大小分级测试**：当输入 footprint 接近 L2 容量时，缓存命中会让数字异常好看——这时要换多个 footprint 看趋势
- **首轮单独留档**：如果首轮特别慢，把它作为「初始化成本」单独记下来，正式统计前扔掉

> **⚠️ 常被忽略的细节**：warmup 之后**也要 synchronize**，否则 warmup 的任务可能还没真正在 GPU 上跑完，计时起点就被污染了。

## 5.3 重复运行与统计

为什么「跑一次」永远不够？

单次结果可能只是偶然：一次后台任务、一次调度抖动、一次缓存命中，都足以让数字漂移。所以可信 benchmark 的第二个支柱是 **repeat（重复多次）**，并对这组时间做统计汇总，而不是只挑一个数字报告。

一个最小 benchmark 的流程：

> 1. 准备固定输入  
> 2. 记录硬件、软件版本和参数  
> 3. 运行若干次 warmup  
> 4. 等待设备完成（synchronize）  
> 5. 重复计时多次  
> 6. 每次计时都确保测量范围一致  
> 7. 汇总 mean / median / min / 波动  
> 8. 保存原始输出和结论

**一个 benchmark 如果波动很大，你应该先修测量方法，而不是急着优化代码。**

## Parameter

### 通用 benchmark 参数

| 参数 | 含义 | 推荐值 |
|------|------|--------|
| `--warmup` | 热身次数，消除首轮开销 | 20 |
| `--repeat` | 正式计时重复次数 | 200 |
| `--shape` | 输入张量形状 | 4096,4096 |
| `--dtype` | 数据类型 | fp16 / fp32 |

### 统计量说明

`mean`、`median`、`min` 这几个统计量各有用处：

- **min**: 最好的一次，常用来观察较少受外部干扰时的能力（估算带宽 / 算力上限时常以 min 为分母）
- **median**: 中位数，更能代表多数情况下的表现
- **mean**: 平均值，容易受异常慢的一次影响
- **p95**: 95 分位数，观察尾延迟有多差
- **std**: 标准差，判断这个 benchmark 是否稳定
- **波动范围 / p99**: 告诉你这个 benchmark 是否稳定

不要只相信一个数字。看到「改完只快了一点点」时，先问：repeat 够吗？median 和 std 怎么变？是不是落在了正常波动范围内？

## 5.4 GPU event 计时

本章最关键的技术细节：**为什么不能用 `time.time()` / wall clock 量 kernel，而要用 GPU event**。

GPU 任务通常是**异步提交**的：你在 host 端调用 `torch.softmax(x)` 或 `kernel<<<...>>>()` 时，CPU 只是把命令塞进队列就立刻返回了，kernel 真正执行完可能还要等一会儿。如果你用 `time.time()` 在调用前后取差，量到的多半是「CPU 把命令塞进队列花了多久」，而不是「GPU 算了多久」——这就是直觉表里「GPU 时间很短，说明程序很快」那条坑的来源。

正确的做法是用 GPU 自己的计时机制，在设备时间线上打两个事件，再算它们之间的间隔。下面的代码骨架演示的就是这套最小流程：

> **warmup → record event → repeat → synchronize → elapsed**

## Execution

### 步骤1：定位仓库根目录

所有路径均相对于仓库根目录。先确认当前位置：

In [ ]:
import os
import subprocess
from pathlib import Path

# 定位仓库根目录（notebooks/part1-profiling/ -> 向上两级）
REPO_ROOT = Path.cwd().resolve().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
os.chdir(REPO_ROOT)
print(f"仓库根目录: {REPO_ROOT}")
print(f"当前工作目录: {Path.cwd()}")

### 步骤2：激活 ROCm 环境

In [ ]:
# 激活 ROCm 环境（如果尚未激活）
activate_script = REPO_ROOT / "code/part1-profiling/activate-rocm.sh"
if activate_script.exists():
    print(f"ROCm 激活脚本: {activate_script}")
    print("请在终端执行: source code/part1-profiling/activate-rocm.sh")
else:
    print("警告: 未找到 ROCm 激活脚本")

### 步骤3：骨架 A — PyTorch vector add（GPU event 计时）

**骨架 A** 是最常见的入口：你想知道某个 PyTorch 算子在某个 shape 上有多快。

要点：

- 用 `torch.cuda.Event` 而不是 `time.time()`——前者计的是 GPU 时间，后者会被异步 launch 误导
- warmup 至少 20 次，并在 warmup 后 `synchronize`
- 同时输出 mean / median / min / p95 / std，单一数字不够

下面的代码演示 GPU event 计时和统计量输出，将 `bench_torch_op` 的短逻辑直接放在 Notebook 中：

In [ ]:
import torch
import statistics

def bench_torch_op(shape, dtype, repeats=200, warmup=20):
    """骨架 A：PyTorch 算子计时（示例：vector add）"""
    x = torch.randn(*shape, dtype=dtype, device="cuda")
    y = torch.randn(*shape, dtype=dtype, device="cuda")
    
    # warmup：让 JIT、cache、clock 进入稳定状态
    for _ in range(warmup):
        z = x + y
    torch.cuda.synchronize()
    
    # 用 GPU event 计时，不要用 time.time()
    starts = [torch.cuda.Event(enable_timing=True) for _ in range(repeats)]
    ends = [torch.cuda.Event(enable_timing=True) for _ in range(repeats)]
    for i in range(repeats):
        starts[i].record()
        z = x + y
        ends[i].record()
    torch.cuda.synchronize()
    
    times_ms = [s.elapsed_time(e) for s, e in zip(starts, ends)]
    return {
        "mean": statistics.mean(times_ms),
        "median": statistics.median(times_ms),
        "min": min(times_ms),
        "p95": sorted(times_ms)[int(len(times_ms) * 0.95)],
        "std": statistics.pstdev(times_ms),
    }

# 执行 benchmark
if torch.cuda.is_available():
    shape = (4096, 4096)
    for dt_name, dt in [("fp32", torch.float32), ("fp16", torch.float16)]:
        stats = bench_torch_op(shape, dt)
        print(f"\n{dt_name} vector add @ {shape}:")
        print(f"  min={stats['min']:.3f} ms, median={stats['median']:.3f} ms")
        print(f"  mean={stats['mean']:.3f} ms, p95={stats['p95']:.3f} ms, std={stats['std']:.4f} ms")
else:
    print("GPU 不可用，跳过 benchmark")

### 步骤4：骨架 B — Triton kernel 计时与有效带宽

**骨架 B** 针对单个 kernel 的 micro-benchmark，重点是**用 bytes / ops 估算反推有效带宽或算力**，再和硬件峰值比较，判断「离极限多远」。

要点：

- **用一个公式把时间换算成 BW（或 TFLOPS）**——单看时间没法判断「离峰值多远」
- 当 `n` 远大于 L2 容量时，`eff_bw` 应该逼近 GDDR6 实测带宽，否则 benchmark 流程本身有问题
- 同样的骨架可以替换 kernel 来测 GEMM、softmax 等，只要把「搬运 bytes / 计算 ops」的公式换掉

本节将 `bench_triton_copy` 的短逻辑直接放在 Notebook 中，便于逐项解释参数：

In [ ]:
try:
    import triton
    import triton.language as tl
    
    @triton.jit
    def copy_kernel(x_ptr, y_ptr, n, BLOCK: tl.constexpr):
        pid = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < n
        tl.store(y_ptr + offs, tl.load(x_ptr + offs, mask=mask), mask=mask)
    
    def bench_triton_copy(n, repeats=200, warmup=20, block=1024):
        """骨架 B：Triton kernel 计时与有效带宽估算"""
        x = torch.empty(n, dtype=torch.float32, device="cuda")
        y = torch.empty_like(x)
        grid = ((n + block - 1) // block,)
        
        for _ in range(warmup):
            copy_kernel[grid](x, y, n, BLOCK=block)
        torch.cuda.synchronize()
        
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(repeats):
            copy_kernel[grid](x, y, n, BLOCK=block)
        e.record()
        torch.cuda.synchronize()
        ms = s.elapsed_time(e)
        
        # 每次迭代搬运 2 * n * 4 字节（一读一写 fp32）
        total_bytes = 2 * n * 4 * repeats
        eff_bw = total_bytes / (ms * 1e-3) / 1e9  # GB/s
        return ms, eff_bw
    
    # 执行 benchmark
    if torch.cuda.is_available():
        print("\nTriton vector copy (float32):")
        for mib in [8, 64, 256]:
            n = mib * 1024 * 1024 // 4
            ms, gbps = bench_triton_copy(n)
            print(f"  {mib:3d} MiB: time={ms:.3f} ms, eff_bw={gbps:.2f} GB/s")
    else:
        print("GPU 不可用，跳过 Triton benchmark")
        
except ImportError:
    print("Triton 不可用，跳过骨架 B")

### 补充：骨架 C — HIP event 计时

有时你需要绕过 Python 直接量 HIP kernel。HIP event 计时的最小骨架如下（C++ 代码，不在 Notebook 中执行）：

```cpp
// code/part1-profiling/chapter5/bench_hip.cpp
// 用法：hipcc -O3 bench_hip.cpp -o bench_hip && ./bench_hip
#include <hip/hip_runtime.h>
#include <cstdio>

__global__ void my_kernel(float* x, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) x[i] = x[i] * 2.0f + 1.0f;
}

int main() {
    const int n = 1 << 24;
    const int warmup = 20, repeats = 200;
    float* d;
    hipMalloc(&d, n * sizeof(float));

    int block = 256;
    int grid  = (n + block - 1) / block;

    for (int i = 0; i < warmup; ++i)
        my_kernel<<<grid, block>>>(d, n);
    hipDeviceSynchronize();

    hipEvent_t s, e;
    hipEventCreate(&s); hipEventCreate(&e);
    hipEventRecord(s);
    for (int i = 0; i < repeats; ++i)
        my_kernel<<<grid, block>>>(d, n);
    hipEventRecord(e);
    hipEventSynchronize(e);

    float ms = 0.f;
    hipEventElapsedTime(&ms, s, e);
    printf("avg per launch = %.4f ms\n", ms / repeats);

    hipFree(d);
    return 0;
}
```

要点：

- `hipEvent_t` 的精度足够量到 μs 级 kernel
- 一定要 `hipEventSynchronize` 之后再读 elapsed time，否则 host 还在拿着 stale 值
- 第 6 章会沿用这个计时骨架，再用 `rocprofv3` 核对 kernel 时间

### 实测数字（Radeon RX 9070 XT + ROCm 7.13）

下表是用 `bench_ch4.py`（综合了骨架 A / B 两段流程）在 9070XT（gfx1201 / ROCm 7.13 / 原生 Ubuntu 24.04）上跑出来的实测值：

| 实验 | 算子 / 公式 | shape / dtype | 时延（median / min） | 有效带宽 | 算术强度 |
| ---- | ---- | ---- | ----: | ----: | ----: |
| 骨架 A — PyTorch vector add | `c = a + b` | 4096² / fp32 | 0.337 / 0.335 ms | 600.8 GB/s | ~0.083 FLOP/B |
| 骨架 A — PyTorch vector add | `c = a + b` | 4096² / fp16 | 0.173 / 0.171 ms | 587.3 GB/s | ~0.17 FLOP/B |
| 骨架 B — Triton vector copy | `y = x` | 8 MiB（pair） | 0.020 ms（min） | 785.1 GB/s | — |
| 骨架 B — Triton vector copy | `y = x` | 64 MiB（pair） | 0.223 ms（min） | 572.8 GB/s | — |
| 骨架 B — Triton vector copy | `y = x` | 256 MiB（pair） | 0.900 ms（min） | 568.6 GB/s | — |

读这张表的关键点：

- **memory-bound 算子的「快」上限就是带宽**：vector add 在 fp32 / fp16 下有效带宽几乎一致（600.8 vs 587.3 GB/s），但 fp16 的时间只有 fp32 的约一半（0.171 vs 0.335 ms）——fp16 真省到的是 byte 数，算术强度跟着翻倍
- **vector copy 的 footprint 扫描能画出 cache 层级**：8 MiB 时有效带宽冲到 785.1 GB/s（L2 命中区），64 MiB 以后跌到 ~570 GB/s 并稳定——这就是踩进 GDDR6 平台后的真实带宽

> 时延口径：vector add 用 GPU event 逐次计时，同时报告 min 与 median；vector copy 用「一段 event 覆盖 200 次连续 launch 后求平均」。有效带宽口径：vector add 按 `3 × elems × dtype` 字节（两读一写），vector copy 按 `2 × footprint` 字节（一读一写）。

> 太小的输入（几 MiB 以下）测出来的不是带宽峰值，是 launch overhead——每次 copy 真正干活只有几 μs，被启动开销稀释。要看 cache 层级必须跑 footprint 扫描，而不是只跑一个 size。

### 步骤5：执行综合 benchmark（bench_ch4.py）

运行完整的第4章 benchmark 骨架，输出所有统计量：

In [ ]:
bench_ch4_script = REPO_ROOT / "code/part1-profiling/chapter5/bench_ch4.py"

if bench_ch4_script.exists():
    print(f"执行: python {bench_ch4_script.relative_to(REPO_ROOT)}")
    result = subprocess.run(
        ["python", str(bench_ch4_script)],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        timeout=120
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
else:
    print(f"未找到脚本: {bench_ch4_script}")

## 5.5 避免测量陷阱

这一节专门讲「看起来变快了，但其实不一定」的情况——也就是常说的**伪优化**。伪优化最麻烦的地方在于，它会给你一种很强的成就感：数字变好了，代码也改了，好像问题解决了。但如果测量方式不可靠，后面换输入、换机器、换版本时，结果很可能消失。

| 现象 | 可能原因 | 应对方式 |
| ---- | ---- | ---- |
| 第一次很慢，后面明显变快 | 首轮包含初始化、编译、缓存准备 | 单独记录首轮，正式统计前 warmup |
| 改完只快了一点 | 可能是正常波动 | 增加 repeat，看 median 和 std |
| GPU 计时几乎为零 | 没有等待 GPU 完成 | 使用 GPU event 或显式 synchronize |
| 小输入特别快 | 可能主要测到 launch overhead 或缓存效果 | 用多个输入规模观察趋势 |
| 单个 kernel 快了，但端到端没变化 | 瓶颈不在这个 kernel | 先看它在总耗时中占比 |
| 前后版本差异很大 | 同时改了多个变量 | 一次只改一个变量，保留对照组 |
| 带宽或 FLOPS 看起来异常高 | 数据量模型或计时范围不一致 | 重新核对 bytes / ops 的估算口径 |
| 改 dtype 之后吞吐翻倍 | 可能只是 Tensor 数量变了一半 | 把 byte 数和 ops 数都重新算 |
| 关掉某个 print / log 后变慢 | print 把 host 卡住，意外起到了同步效果 | 计时范围要明确包含或排除日志 |
| 第二次运行就一直很快 | 数据已被 L2 / GDDR6 预热 | 按工作集大小分级测试 |

避免伪优化的核心方法：**让实验可复查。** 一次好的性能实验，至少应该留下：输入规模、数据类型、硬件和软件版本、运行命令、原始输出、统计方式、结论。

> **如果你现在只记住一条：优化前先建立可信 baseline。没有 baseline，后面所有「更快了」都没有参照物。**

### 步骤6：统计量示例与解读

演示如何解读一组时间数据——包括变异系数（CV = std / mean）的稳定性判断：

In [ ]:
import statistics

# 模拟一组测量数据（单位：ms）
times_ms = [0.335, 0.337, 0.336, 0.338, 0.340, 0.339, 0.337, 0.336, 0.338, 0.341]

stats = {
    "min": min(times_ms),
    "median": statistics.median(times_ms),
    "mean": statistics.mean(times_ms),
    "p95": sorted(times_ms)[int(len(times_ms) * 0.95)],
    "std": statistics.pstdev(times_ms),
}

print("统计量示例:")
print(f"  原始数据: {times_ms}")
print(f"  min={stats['min']:.3f} ms  (最短时间，估算峰值时常用)")
print(f"  median={stats['median']:.3f} ms  (中位数，代表多数情况)")
print(f"  mean={stats['mean']:.3f} ms  (平均值，易受异常影响)")
print(f"  p95={stats['p95']:.3f} ms  (95分位数，观察尾延迟)")
print(f"  std={stats['std']:.4f} ms  (标准差，判断波动)")

# 判断稳定性
cv = stats['std'] / stats['mean']  # 变异系数
print(f"\n变异系数 (CV) = {cv:.4f}")
if cv < 0.05:
    print("  -> 波动很小，测量稳定")
elif cv < 0.10:
    print("  -> 波动可接受")
else:
    print("  -> 波动较大，需检查测量方法")

## 5.6 可信 benchmark 的检查清单

把前面几节的要点收成一份清单——每次开一组实验前过一遍，能挡掉大部分伪优化：

| 检查项 | 为什么重要 | 常见错误 |
| ---- | ---- | ---- |
| 固定输入规模 | 输入变了，时间自然会变 | 前后对比时 shape 不一致 |
| 固定数据类型 | fp32、fp16、bf16 的计算路径不同 | 只说「快了」，不说 dtype |
| 区分初始化和正式计时 | 第一次运行可能包含加载、编译、缓存准备 | 把首轮初始化当成稳定性能 |
| warmup | 让缓存、JIT、设备状态进入稳定状态 | 第一轮特别慢，直接拿来平均 |
| repeat | 单次结果可能只是偶然 | 只跑一次就下结论 |
| synchronize | GPU 任务常常异步提交 | 只量到 CPU 提交时间 |
| 计时器选择 | wall clock vs GPU event 精度差异大 | 用 `time.time()` 量微秒级 kernel |
| 锁定时钟 / 后台干扰 | 频率波动会污染数据 | 后台跑着别的 GPU 任务 |
| 记录环境 | 后续复查需要硬件、驱动、框架版本 | 只有一个数字，没有上下文 |
| 只改一个变量 | 才知道是谁带来变化 | 同时改 shape、dtype、实现和参数 |

把这张表和三段骨架配合起来用：骨架管「怎么测」，清单管「测得对不对」。两者都到位，一次 benchmark 才值得相信。

## Expected Output / Interpretation

### 骨架 A 预期输出（PyTorch vector add）

在 9070XT (gfx1201) + ROCm 7.13 上，预期看到：

```
fp32 vector add @ (4096, 4096):
  min=0.335 ms, median=0.337 ms
  mean=0.338 ms, p95=0.341 ms, std=0.0020 ms

fp16 vector add @ (4096, 4096):
  min=0.171 ms, median=0.173 ms
  mean=0.174 ms, p95=0.177 ms, std=0.0018 ms
```

**解读**：
- fp16 时间约为 fp32 的一半（0.171 vs 0.335 ms）
- 但有效带宽几乎一致（~600 GB/s），说明都是 memory-bound
- std 很小（< 0.002 ms），说明测量稳定

### 骨架 B 预期输出（Triton vector copy）

```
Triton vector copy (float32):
    8 MiB: time=0.020 ms, eff_bw=785.1 GB/s
   64 MiB: time=0.223 ms, eff_bw=572.8 GB/s
  256 MiB: time=0.900 ms, eff_bw=568.6 GB/s
```

**解读**：
- 8 MiB 时带宽很高（785 GB/s），数据主要在 L2 cache
- 64 MiB 以后跌到 ~570 GB/s 并稳定，这是 GDDR6 真实带宽
- 这条曲线是后续所有 memory-bound 算子的参照线

### 关键观察点

1. **首轮特别慢**：warmup 前的第一次包含编译、缓存准备、时钟爬升
2. **median vs min**：median 更稳定，min 用于估算峰值
3. **footprint 扫描**：小输入测到 cache，大输入测到 GDDR6
4. **波动范围**：std / mean < 5% 说明测量可信

## 本章小结

- 性能优化不是从改代码开始，而是从定义问题和设计测量开始；**没有稳定测量，就没有可靠优化**。
- 首轮的一次性开销（编译、冷缓存、爬频）要在 warmup 里消掉；正式统计只看稳态段，warmup 之后记得 synchronize。
- 单次结果不可信，要 repeat 多次并汇总 mean / median / min / p95 / std；波动大时先修测量方法，而不是急着优化代码。
- GPU 任务是异步提交的，必须用 GPU event（`torch.cuda.Event` / `hipEvent_t`）而不是 `time.time()` 量 kernel 真实耗时；本章给出 PyTorch / Triton / HIP 三段最小骨架。
- 伪优化有很多伪装（缓存命中、launch overhead、改 dtype 只省了 byte、print 意外同步……），核心对策是「让实验可复查」和「先建立可信 baseline」。
- 下一章会用两个 vector add 版本，把本章的 benchmark 流程和 `rocprofv3` 串成「量准 → 找到慢点 → 验证」的完整路线。

## Pass Criteria

本章操作通过标准：

### 必须满足

1. ✅ 能执行骨架 A（PyTorch vector add），输出 min / median / mean / p95 / std
2. ✅ 能执行骨架 B（Triton vector copy），输出有效带宽
3. ✅ 理解 warmup 的作用：消除首轮编译、缓存、时钟爬升开销
4. ✅ 理解 repeat 的作用：单次结果不可信，需要统计汇总
5. ✅ 理解 GPU event vs wall clock：异步提交时必须用 GPU event
6. ✅ 能解读统计量：min 估算峰值，median 代表多数，std 判断波动

### 推荐完成

7. ⭐ 运行完整的 `bench_ch4.py`，对比 fp32 / fp16 的时间和带宽
8. ⭐ 观察 footprint 扫描曲线（8 / 64 / 256 MiB），识别 L2 vs GDDR6
9. ⭐ 计算变异系数（CV = std / mean），判断测量是否稳定

### 常见问题排查

| 现象 | 可能原因 | 解决方法 |
|------|----------|----------|
| GPU 不可用 | ROCm 未激活 | `source code/part1-profiling/activate-rocm.sh` |
| Triton 导入失败 | 缺少 Python.h | `sudo apt install python3-dev` |
| 时间几乎为零 | 没有 synchronize | 检查 `torch.cuda.synchronize()` |
| 波动很大 | 后台负载 | 关闭其他 GPU 任务，增加 repeat |
| 首轮特别慢 | 正常现象 | warmup 后单独记录，不混入统计 |

---

## 延伸阅读

- [HIP Performance Guidelines](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/performance_guidelines.html)
- [HIP Programming Guide](https://rocm.docs.amd.com/projects/HIP/en/latest/) — HIP 编程模型与计时 API 入口
- [PyTorch Profiler 文档](https://docs.pytorch.org/docs/stable/profiler.html)
- [ROCm Documentation](https://rocm.docs.amd.com/)

**下一章**: [第6章 用 rocprof 找到慢在哪里](./chapter6.ipynb)